In [1]:
# Install the core, the community tools, and the Groq integration
!pip install -U langchain langchain-community langchain-groq pypdf python-dotenv

In [2]:
# 1. Install dependencies
!pip install -q langchain langchain-groq pypdf python-dotenv langchain-community

import os

# 2. Setup folders
folders = ["data/resumes", "prompts", "chains", "utils"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# 3. Set Environment Variables
# REPLACE THESE WITH YOUR KEYS
os.environ["GROQ_API_KEY"] = "Your Groq API Key"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "Your LangChain API Key"
os.environ["LANGCHAIN_PROJECT"] = "Innomatics_AI_Resume_Screener"

print("✅ Environment and Folders Ready.")

✅ Environment and Folders Ready.


In [3]:
%%writefile prompts/templates.py
from langchain_core.prompts import ChatPromptTemplate

def get_screener_prompt():
    template = """
    You are an expert AI Engineering Recruiter. Your task is to analyze the resume and compare it against the Job Description.

    JD: {job_description}
    Resume: {resume_text}

    STRICT JSON FORMATTING RULES:
    1. "candidate_name": Extract the full name.
    2. "score": An integer (0-100).
    3. "extracted_skills": List ONLY the technical AI/ML skills found in the resume.
    4. "explanation": A detailed reason why this candidate received the score based on their experience.

    Return ONLY the JSON object.
    """
    return ChatPromptTemplate.from_template(template)

Overwriting prompts/templates.py


In [4]:
%%writefile utils/pdf_loader.py
from langchain_community.document_loaders import PyPDFLoader

def extract_text_from_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    return " ".join([page.page_content for page in pages])

Overwriting utils/pdf_loader.py


In [5]:
%%writefile chains/screener_chain.py
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from prompts.templates import get_screener_prompt

def get_screener_chain():
    # llama-3.3-70b provides superior technical reasoning for skill extraction
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    parser = JsonOutputParser()
    prompt = get_screener_prompt()
    return prompt | llm | parser

Overwriting chains/screener_chain.py


In [6]:
%%writefile data/jd.txt
Role: AI Engineer
Core Skills: Python, LangChain, RAG, Vector Databases (Chroma/Pinecone), PyTorch/TensorFlow, LLM Fine-tuning, FastAPI, and Docker.

Overwriting data/jd.txt


In [7]:
import importlib
import utils.pdf_loader
import chains.screener_chain

importlib.reload(utils.pdf_loader)
importlib.reload(chains.screener_chain)

from utils.pdf_loader import extract_text_from_pdf
from chains.screener_chain import get_screener_chain

chain = get_screener_chain()
print("✅ Modular system initialized for AI Engineer screening.")

✅ Modular system initialized for AI Engineer screening.


In [8]:
import os

with open("data/jd.txt", "r") as f:
    jd = f.read()

resume_dir = "data/resumes/"
files = [f for f in os.listdir(resume_dir) if f.endswith(".pdf")]

if not files:
    print("⚠️ No PDFs found. Please upload resumes to data/resumes/ in the sidebar!")
else:
    print(f"🚀 Found {len(files)} resumes. Starting Extraction...")
    print("-" * 60)

    for filename in files:
        path = os.path.join(resume_dir, filename)
        resume_text = extract_text_from_pdf(path)

        try:
            # Generate the analysis
            result = chain.invoke({"job_description": jd, "resume_text": resume_text})

            # Print the results
            print(f"📄 FILE: {filename}")
            print(f"👤 NAME: {result.get('candidate_name', 'Unknown')}")
            print(f"📊 SCORE: {result.get('score')}/100")
            print(f"🛠️ SKILLS EXTRACTED: {', '.join(result.get('extracted_skills', []))}")
            print(f"💡 REASONING: {result.get('explanation')}")
            print("-" * 60)

        except Exception as e:
            print(f"❌ Error with {filename}: {e}")

🚀 Found 3 resumes. Starting Extraction...
------------------------------------------------------------
📄 FILE: strong_candidate.pdf
👤 NAME: Ranadeep Kooragayala
📊 SCORE: 70/100
🛠️ SKILLS EXTRACTED: Python, LangChain, RAG, FastAPI, OpenCV, OpenAI
💡 REASONING: The candidate has a strong foundation in AI/ML with skills in Python, LangChain, RAG, and FastAPI, which are core requirements for the AI Engineer role. Additionally, their experience as an Advanced GenAI Intern at Innomatics Research Labs, where they developed AI-powered systems using LangChain and LLMs, is highly relevant. However, the candidate lacks experience with Vector Databases (Chroma/Pinecone), PyTorch/TensorFlow, and LLM Fine-tuning, which are also essential skills for the role. Their projects, such as the Library Management System using FastAPI, demonstrate their ability to apply AI/ML concepts to real-world problems. Overall, the candidate has a good balance of technical skills and relevant experience, but requires add